# CS-781 Capstone - Phase 3: Rhetorical Role prediction (RR) - one notebook, every technique

**Model (same for every experiment, so the comparison is about the adaptation technique and the backbone):**
a *hierarchical sentence tagger* on top of your Phase-1 legal LM.

1. Consecutive sentences of a document are packed into chunks of <= `CHUNK_TOK` tokens, so every sentence is encoded together with its neighbours (local context).
2. The **Phase-1 backbone** (GQA / MLA / RetNet decoders, or the enc-dec *encoder*) encodes each chunk. Decoder-only backbones are causal, so each chunk is read twice (`[c, c]`) and features come from the second copy - every token sees the whole chunk. The encoder gets a key-padding mask.
3. Each sentence is pooled (masked mean + learned attention pooling), then a **domain embedding** (CL / IT) and **relative-position features** (rhetorical roles depend strongly on where a sentence sits in a judgment) are added.
4. A 2-layer **BiGRU runs over all sentences of the document** (document-level context), then a linear layer (optionally a CRF) gives one of the 13 roles per sentence.
5. Class-weighted cross-entropy for the rare roles, early stopping on dev **macro-F1**, then **per-class logit-bias tuning on dev** (macro-F1 rewards getting rare roles partly right). A 2-fold estimate shows how much of that gain is real.

**Techniques** (all written from scratch in `rr_peft.py`, no `peft` library). Set `METHOD` and `BACKBONE_ARCH` and run top to bottom:

| `METHOD` | technique | suggested backbone for your 6 submissions |
|---|---|---|
| `lorma` | LoRMA+ (multiplicative low-rank, **compulsory**) | `gqa` |
| `rslora` | rank-stabilised LoRA (r=64, alpha/sqrt(r)) | `retnet` |
| `ia3` | IA3 (rescale keys, values, FFN activations) | `mla` |
| `lora` | LoRA | `encdec` |
| `adalora` | AdaLoRA (SVD form + importance-based rank pruning) | `retnet` or `gqa` |
| `kd` | **knowledge distillation** (mandatory): teacher = average of earlier runs' probabilities, student = half-depth Phase-1 model | `gqa` |
| `prompt`, `qlora` | prompt tuning, QLoRA (4-bit NF4 frozen weights + LoRA) | extras |

The backbone itself stays frozen in every PEFT run (only the adapter, the pooling/BiGRU head and the classifier train). Every run writes `rr_<method>_<arch>/` with `submission.zip`, `results.json` and the dev/test/train **probabilities** - the ensemble notebook and the KD run read those.

Optional `TAPT = True`: task-adaptive pre-training of the Phase-1 model on the *unlabelled* text of the RR train/dev documents before fine-tuning (same objective as Phase 1, official data only). Ask a TA once whether this counts as "starting from your Phase-1 model" before you submit a TAPT run.

## 1. Hugging Face login

In [ ]:
import os
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("No HF token found - continuing anonymously (works only if the dataset is public).")

## 2. GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i), "| capability", torch.cuda.get_device_capability(i))

## 3. Configuration - change `METHOD` / `BACKBONE_ARCH` per run

In [ ]:
import os, sys, json, math, time, random, glob, copy, importlib
from pathlib import Path
import numpy as np
import torch

METHOD        = "lorma"     # lorma | rslora | ia3 | lora | adalora | kd | prompt | qlora
BACKBONE_ARCH = "gqa"       # gqa | mla | encdec | retnet   (for kd: the architecture of teacher-size model the student is cut from)
P1_EXPORT_DIR = None        # None -> auto-search /kaggle/input/**/export_<arch>/
RUN_TAG       = ""          # optional suffix for the output folder (e.g. "_crf", "_tapt")

# ---- data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
NORMALIZE_WHITESPACE = True          # must match Phase-1 training (True in all four Phase-1 notebooks)
SEED = 42

# ---- input packing / context
MAX_SENT_TOK = 128     # longer sentences keep head + short tail
CHUNK_TOK    = 320     # sentences packed into chunks of this many tokens (decoders read 2x this because of echo)
CHUNK_BS     = 8       # chunks per backbone call
MAX_SENTS    = 96      # training window: consecutive sentences per step (inference always uses the whole document)

# ---- head
HCFG = dict(proj=512, gru_hidden=256, gru_layers=2, dropout=0.2, crf=False, echo=True, grad_ckpt=True, chunk_bs=CHUNK_BS)

# ---- optimisation
EPOCHS, PATIENCE = 12, 4
HEAD_LR    = 1e-3
ACCUM      = 4          # windows per optimiser step
WD, WARMUP, GRAD_CLIP = 0.01, 0.06, 1.0
LABEL_SMOOTH     = 0.05
CLASS_WEIGHT_POW = 0.5  # 0 = no class weighting, 1 = full inverse frequency
TUNE_BIAS        = True # per-class logit biases tuned on dev for macro-F1 (both versions of the submission are written)

# ---- optional task-adaptive pre-training (continued Phase-1 objective on RR train+dev text, official data only)
TAPT, TAPT_EPOCHS, TAPT_LR, TAPT_MICRO_BS = False, 2, 1e-4, 2

# ---- per-method hyper-parameters (lr = learning rate of the adapter / backbone parameters)
MCFG_ALL = {
    "lora":    dict(r=16, alpha=32, dropout=0.05, lr=5e-4),
    "rslora":  dict(r=64, alpha=16, dropout=0.05, lr=3e-4),
    "lorma":   dict(r=16, alpha=16, variant="plus", lr=5e-4),
    "ia3":     dict(out_targets=("wk", "wv", "w_ukv"), in_targets=("w_down",), lr=3e-3),
    "adalora": dict(init_r=12, target_r=6, alpha=12, lr=1e-3, orth_reg=0.1, delta_t=5),
    "prompt":  dict(n_prompt=20, lr=3e-3),
    "qlora":   dict(r=16, alpha=32, dropout=0.05, block=64, lr=5e-4),
    # kd: student = Phase-1 model with every other layer kept, fully fine-tuned on alpha*CE + (1-alpha)*T^2*KL(teacher||student)
    "kd":      dict(keep=None, lr=5e-5, alpha=0.3, T=2.0, teacher_dirs=None),
}
MCFG = dict(MCFG_ALL[METHOD])

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WORK = ROOT / f"rr_{METHOD}_{BACKBONE_ARCH}{RUN_TAG}"
WORK.mkdir(parents=True, exist_ok=True)
print("device:", device, "| output folder:", WORK, "| method cfg:", MCFG)

## 4. Library (`rr_lib.py`) and the from-scratch PEFT methods (`rr_peft.py`)

In [ ]:
%%writefile rr_lib.py
"""CS-781 Phase 3 (Rhetorical Role prediction) helper library.

Hierarchical sentence tagger on top of a Phase-1 legal LM:
  sentences of a document --packed--> chunks of <= CHUNK_TOK tokens (neighbouring sentences = local context)
  --> Phase-1 backbone (+ from-scratch PEFT modules, see rr_peft.py) --> token features
  --> per-sentence pooling (masked mean + learned attention pooling)
  --> + domain embedding (CL / IT) + relative-position features
  --> 2-layer BiGRU across ALL sentences of the document (document-level context)
  --> linear (optional CRF) --> one of the 13 rhetorical roles per sentence.
Decoder-only backbones are causal, so each chunk is read twice ([c, c]) and features come from the second copy
(every token sees the whole chunk). The enc-dec backbone uses its bidirectional encoder with a key-padding mask.
"""
import os, sys, re, json, math, time, glob, shutil, random, hashlib, types, zipfile, importlib.util, contextlib
from collections import Counter
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

PAD_ID, EOS_ID = 1, 0
DOMAINS = ("CL", "IT")
N_POS = 9


# ====================================================================== loading Phase-1 models (same as Phase 2)
def load_module_from_file(path, name):
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def file_hash(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()[:10]


def find_p1_export(arch, roots=("/kaggle/input", "/kaggle/working", ".")):
    hits = []
    for r in roots:
        hits += glob.glob(f"{r}/**/export_{arch}/config.json", recursive=True)
    return Path(hits[0]).parent if hits else None


def load_p1(export_dir):
    """Returns (arch_module, cfg_dict, backbone_with_phase1_weights)."""
    export_dir = Path(export_dir)
    cj = json.load(open(export_dir / "config.json"))
    mod = load_module_from_file(export_dir / "legal_arch.py",
                                f"legal_arch_{cj['arch']}_{file_hash(export_dir / 'legal_arch.py')}")
    bb = mod.build_model(mod.ArchConfig(**cj["cfg"]))
    bb.load_state_dict(torch.load(export_dir / "model.pt", map_location="cpu", weights_only=False))
    return mod, cj["cfg"], bb


def blocks_of(bb):
    return bb.blocks if bb.kind == "decoder" else bb.enc_blocks


def final_norm_of(bb):
    return bb.norm_f if bb.kind == "decoder" else bb.enc_norm


def strip_backbone(bb):
    """Keep what RR needs: embeddings + decoder-only stack, or only the ENCODER of the enc-dec model."""
    if bb.kind == "encdec":
        del bb.dec_blocks
        del bb.dec_norm
    bb.lm_head = nn.Identity()
    return bb


def replace_modules(bb, match, factory):
    n = 0
    for blk in blocks_of(bb):
        for name, mod in list(blk.named_modules()):
            if name and match(name, mod):
                parent_name, _, leaf = name.rpartition(".")
                parent = blk.get_submodule(parent_name) if parent_name else blk
                setattr(parent, leaf, factory(mod))
                n += 1
    return n


def replace_linears(bb, targets, factory):
    targets = set(targets)
    return replace_modules(bb, lambda n, m: isinstance(m, nn.Linear) and n.split(".")[-1] in targets, factory)


def count_params(m):
    tot = sum(p.numel() for p in m.parameters()) + sum(b.numel() for n, b in m.named_buffers() if "packed" in n) * 2
    tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return tot, tr


def make_student_bb(mod, cfg, full_bb, keep):
    """Smaller student initialised from Phase-1 weights: keep only the layers listed in `keep`."""
    cfg = dict(cfg)
    if full_bb.kind == "decoder":
        cfg["n_layers"] = len(keep)
    else:
        cfg["n_enc_layers"] = len(keep)
        cfg["n_dec_layers"] = 1
    st = mod.build_model(mod.ArchConfig(**cfg))
    st.tok_emb.load_state_dict(full_bb.tok_emb.state_dict())
    final_norm_of(st).load_state_dict(final_norm_of(full_bb).state_dict())
    for i, j in enumerate(keep):
        blocks_of(st)[i].load_state_dict(blocks_of(full_bb)[j].state_dict())
    return st, cfg


def max_positions(bb):
    rc = getattr(bb, "rope_cos", None)
    if rc is not None:
        return int(rc.shape[0])
    c = bb.cfg
    return int(getattr(c, "max_seq_len", getattr(c, "enc_max", 1024)))


# ====================================================================== key-padding mask for the enc-dec encoder
# The Phase-1 encoder self-attention has no padding mask (it never saw padding). Batches of chunks of different
# lengths are padded, so we patch the encoder attention to accept a key-padding mask (set per call via _ENC_MASK).
_ENC_MASK = [None]


def _masked_enc_attn_forward(self, x, cos, sin, cache=None):
    arch = sys.modules[type(self).__module__]
    B, T, _ = x.shape
    q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    k = self.wk(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    v = self.wv(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    q = arch.apply_rope(q, cos[:T], sin[:T])
    k = arch.apply_rope(k, cos[:T], sin[:T])
    m = _ENC_MASK[0]
    attn_mask = None if m is None else m[:, None, None, :T]
    o = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
    return self.wo(o.transpose(1, 2).reshape(B, T, -1))


def patch_encoder_padding_mask(bb):
    if bb.kind != "encdec":
        return 0
    for blk in bb.enc_blocks:
        blk.attn.forward = types.MethodType(_masked_enc_attn_forward, blk.attn)
    return len(bb.enc_blocks)


# ====================================================================== data
class RRDoc:
    def __init__(self, doc_id, sents, labels, domain):
        self.doc_id, self.sents, self.labels, self.domain = doc_id, sents, labels, domain
        self.tok, self.y, self.pos = None, None, None


def load_label_vocab(obj):
    """Same rules as evaluate_phase3.py: list, label->id or id->label."""
    if isinstance(obj, list):
        return [str(x) for x in obj]
    if isinstance(obj, dict) and obj:
        vals = list(obj.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            return [k for k, _ in sorted(obj.items(), key=lambda kv: kv[1])]
        if all(isinstance(v, str) for v in vals):
            try:
                return [v for _, v in sorted(obj.items(), key=lambda kv: int(kv[0]))]
            except ValueError:
                return vals
        for k in ("labels", "label_vocab", "vocab"):
            if k in obj:
                return load_label_vocab(obj[k])
    raise ValueError(f"unsupported label_vocab format: {str(obj)[:200]}")


def _doc_items(raw):
    """Yields (doc_id, record) for the formats we may meet: list of docs, {id: doc}, or a wrapper {'data': [...]}."""
    if isinstance(raw, dict):
        if len(raw) == 1 and isinstance(next(iter(raw.values())), list) and next(iter(raw)) in ("data", "docs", "documents"):
            raw = next(iter(raw.values()))
        elif "text" in raw and isinstance(raw.get("text"), list) and ("id" in raw or "labels" in raw):
            raw = [raw]
    if isinstance(raw, dict):
        for k, v in raw.items():
            yield str(k), v
    else:
        for i, v in enumerate(raw):
            did = v.get("id", v.get("doc_id", v.get("name", i))) if isinstance(v, dict) else i
            yield str(did), v


def parse_rr_file(path, domain, label_names):
    raw = json.load(open(path, encoding="utf-8"))
    docs = []
    for did, v in _doc_items(raw):
        labs = None
        if isinstance(v, list):
            sents = v
        else:
            sents = v.get("text", v.get("sentences", v.get("sents")))
            labs = v.get("labels", v.get("label", v.get("roles")))
        if isinstance(sents, str):
            sents = [sents]
        if sents and isinstance(sents[0], dict):
            if labs is None:
                labs = [s.get("label", s.get("role")) for s in sents]
            sents = [s.get("text", s.get("sentence", "")) for s in sents]
        sents = ["" if s is None else str(s) for s in sents]
        if labs is not None:
            labs = [label_names[l] if isinstance(l, int) else str(l) for l in labs]
            assert len(labs) == len(sents), f"{did}: {len(sents)} sentences but {len(labs)} labels"
        docs.append(RRDoc(did, sents, labs, domain))
    return docs, raw


def tokenize_docs(docs, tokenizer, max_sent_tok, label2id=None, normalize=True):
    fallback = tokenizer.token_to_id(".")
    fallback = 3 if fallback is None else fallback
    n_tr = 0
    for d in docs:
        texts = [" " + (" ".join(s.split()) if normalize else s) for s in d.sents]
        encs = tokenizer.encode_batch(texts) if texts else []
        d.tok = []
        for e in encs:
            ids = list(e.ids)
            if len(ids) > max_sent_tok:                       # keep head + a short tail of very long sentences
                tail = max_sent_tok // 8
                ids = ids[:max_sent_tok - tail] + ids[-tail:]
                n_tr += 1
            d.tok.append(ids or [fallback])
        n = len(d.sents)
        r = np.arange(n) / max(1, n - 1)
        L = np.array([len(t) for t in d.tok], dtype=np.float32)
        idx = np.arange(n)
        d.pos = np.stack([r, r ** 2, np.sin(np.pi * r), np.cos(np.pi * r), np.sin(2 * np.pi * r), np.cos(2 * np.pi * r),
                          np.log1p(L) / 5.0, (idx == 0), (idx == n - 1)], 1).astype(np.float32).reshape(n, N_POS)
        if d.labels is not None and label2id is not None:
            d.y = np.array([label2id[l] for l in d.labels], dtype=np.int64)
    return n_tr


class Window:
    """A run of consecutive sentences [i0, i1) of one document, packed into chunks of <= chunk_tok tokens."""

    def __init__(self, doc, i0, i1, chunk_tok):
        self.doc, self.i0, self.i1 = doc, i0, i1
        chunks, s_chunk, s_a, s_b, cur = [], [], [], [], []
        for i in range(i0, i1):
            t = doc.tok[i]
            if cur and len(cur) + len(t) > chunk_tok:
                chunks.append(cur)
                cur = []
            s_chunk.append(len(chunks))
            s_a.append(len(cur))
            cur = cur + list(t)
            s_b.append(len(cur))
        if cur:
            chunks.append(cur)
        self.chunks = chunks
        self.s_chunk, self.s_a, self.s_b = np.array(s_chunk), np.array(s_a), np.array(s_b)


def epoch_windows(docs, max_sents, chunk_tok, rng):
    wins = []
    for d in docs:
        n = len(d.sents)
        if n == 0:
            continue
        if n <= max_sents:
            spans = [(0, n)]
        else:
            o = rng.randrange(1, max_sents + 1)               # random tiling offset -> different windows each epoch
            spans = [(max(0, s), min(n, s + max_sents)) for s in range(o - max_sents, n, max_sents)]
        wins += [Window(d, a, b, chunk_tok) for a, b in spans if b > a]
    rng.shuffle(wins)
    return wins


# ====================================================================== model
class CRF(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(C, C))
        self.start = nn.Parameter(torch.zeros(C))
        self.end = nn.Parameter(torch.zeros(C))

    def nll(self, em, y):
        n = em.size(0)
        gold = self.start[y[0]] + em[0, y[0]] + self.end[y[-1]]
        if n > 1:
            gold = gold + self.trans[y[:-1], y[1:]].sum() + em[torch.arange(1, n, device=em.device), y[1:]].sum()
        alpha = self.start + em[0]
        for t in range(1, n):
            alpha = torch.logsumexp(alpha[:, None] + self.trans + em[t][None, :], dim=0)
        return torch.logsumexp(alpha + self.end, 0) - gold

    def params_np(self):
        return (self.trans.detach().float().cpu().numpy(), self.start.detach().float().cpu().numpy(),
                self.end.detach().float().cpu().numpy())


def viterbi_np(em, trans, start, end):
    n, C = em.shape
    score = start + em[0]
    back = np.zeros((n, C), dtype=np.int64)
    for t in range(1, n):
        s = score[:, None] + trans + em[t][None, :]
        back[t] = s.argmax(0)
        score = s.max(0)
    best = [int((score + end).argmax())]
    for t in range(n - 1, 0, -1):
        best.append(int(back[t, best[-1]]))
    return best[::-1]


class RRModel(nn.Module):
    def __init__(self, bb, n_labels, hcfg):
        super().__init__()
        self.bb, self.kind = bb, bb.kind
        self.echo = bool(hcfg.get("echo", True)) and bb.kind == "decoder"
        self.n_prompt = int(getattr(bb, "_n_prompt", 0))
        self.prompt_base = int(getattr(bb, "_prompt_base", 0))
        self.grad_ckpt = bool(hcfg.get("grad_ckpt", True))
        self.chunk_bs = int(hcfg.get("chunk_bs", 8))
        d, P, H, dr = bb.cfg.d_model, hcfg["proj"], hcfg["gru_hidden"], hcfg["dropout"]
        self.att = nn.Linear(d, 1)
        self.proj = nn.Sequential(nn.LayerNorm(2 * d), nn.Linear(2 * d, P), nn.GELU(), nn.Dropout(dr))
        self.dom = nn.Embedding(len(DOMAINS), P)
        nn.init.zeros_(self.dom.weight)
        self.posp = nn.Linear(N_POS, P)
        self.gru = nn.GRU(P, H, num_layers=hcfg["gru_layers"], batch_first=True, bidirectional=True,
                          dropout=dr if hcfg["gru_layers"] > 1 else 0.0)
        self.out = nn.Sequential(nn.Dropout(dr), nn.Linear(2 * H + P, n_labels))
        self.crf = CRF(n_labels) if hcfg.get("crf") else None

    # ---- backbone over a batch of chunks
    def _bb_forward(self, x, valid):
        if self.kind == "encdec":
            _ENC_MASK[0] = valid
            try:
                return self.bb.encode(x)
            finally:
                _ENC_MASK[0] = None
        return self.bb.hidden(x)

    def chunk_features(self, chunks):
        dev = self.att.weight.device
        P = self.n_prompt
        pre = list(range(self.prompt_base, self.prompt_base + P))
        rows, starts = [], []
        for c in chunks:
            if self.echo:
                rows.append(pre + c + c); starts.append(P + len(c))
            else:
                rows.append(pre + c); starts.append(P)
        S, B = max(map(len, rows)), len(rows)
        x = torch.full((B, S), PAD_ID, dtype=torch.long)
        valid = torch.zeros((B, S), dtype=torch.bool)
        for b, r in enumerate(rows):
            x[b, :len(r)] = torch.tensor(r, dtype=torch.long)
            valid[b, :len(r)] = True
        x, valid = x.to(dev), valid.to(dev)
        if self.grad_ckpt and self.training and torch.is_grad_enabled():
            h = checkpoint(self._bb_forward, x, valid, use_reentrant=False)
        else:
            h = self._bb_forward(x, valid)
        Lm = max(map(len, chunks))
        idx = (torch.tensor(starts, device=dev)[:, None] + torch.arange(Lm, device=dev)[None, :]).clamp(max=S - 1)
        return h.gather(1, idx[..., None].expand(-1, -1, h.size(-1)))          # (B, Lm, d), token j of chunk b

    def sentence_vectors(self, win):
        out = []
        for c0 in range(0, len(win.chunks), self.chunk_bs):
            feats = self.chunk_features(win.chunks[c0:c0 + self.chunk_bs])
            sel = np.nonzero((win.s_chunk >= c0) & (win.s_chunk < c0 + self.chunk_bs))[0]
            dev = feats.device
            ci = torch.as_tensor(win.s_chunk[sel] - c0, device=dev)
            a = torch.as_tensor(win.s_a[sel], device=dev)
            b = torch.as_tensor(win.s_b[sel], device=dev)
            Fs = feats[ci]                                                     # (n, Lm, d)
            pos = torch.arange(Fs.size(1), device=dev)
            m = (pos[None] >= a[:, None]) & (pos[None] < b[:, None])
            mf = m.unsqueeze(-1).to(Fs.dtype)
            mean = (Fs * mf).sum(1).float() / mf.sum(1).float().clamp(min=1)
            sc = self.att(Fs).squeeze(-1).float().masked_fill(~m, -1e4)
            attp = (sc.softmax(1).unsqueeze(-1) * Fs.float()).sum(1)
            out.append(torch.cat([mean, attp], -1))
        return torch.cat(out, 0)

    def emissions(self, svec, pos, domain):
        z = self.proj(svec)
        z = z + self.posp(pos) + self.dom.weight[domain]
        h, _ = self.gru(z.unsqueeze(0))
        return self.out(torch.cat([h.squeeze(0), z], -1)).float()

    def forward(self, win):
        sv = self.sentence_vectors(win)
        pos = torch.as_tensor(win.doc.pos[win.i0:win.i1], device=sv.device)
        return self.emissions(sv, pos, win.doc.domain)


# ====================================================================== AMP
def get_amp(device):
    if device == "cuda":
        dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        return (lambda: torch.autocast(device_type="cuda", dtype=dt)), dt
    return (lambda: contextlib.nullcontext()), None


def make_scaler(device, amp_dtype):
    en = device == "cuda" and amp_dtype == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=en)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=en)


# ====================================================================== metrics (same definition as evaluate_phase3.py)
def confusion(y, p, C):
    return np.bincount(np.asarray(y) * C + np.asarray(p), minlength=C * C).reshape(C, C)


def prf(cm):
    tp = np.diag(cm).astype(float)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    p = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    r = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    return p, r, f


def macro_f1(y, p, C):
    return float(prf(confusion(y, p, C))[2].mean())


def report(y, p, labels):
    C = len(labels)
    cm = confusion(y, p, C)
    P, R, Fv = prf(cm)
    sup = cm.sum(1)
    present = sup > 0
    off = [(labels[i], labels[j], int(cm[i, j])) for i in range(C) for j in range(C) if i != j and cm[i, j] > 0]
    off.sort(key=lambda t: -t[2])
    return dict(macro_f1=float(Fv.mean()), macro_p=float(P.mean()), macro_r=float(R.mean()),
                macro_f1_present_labels=float(Fv[present].mean()) if present.any() else 0.0,
                accuracy=float(np.trace(cm) / max(1, cm.sum())), n=int(cm.sum()),
                per_label={labels[i]: dict(p=float(P[i]), r=float(R[i]), f1=float(Fv[i]), support=int(sup[i]),
                                           predicted=int(cm[:, i].sum())) for i in range(C)},
                top_confusions=off[:12], confusion=cm.tolist())


def print_report(rep, title=""):
    print(f"{title} macro-F1 {rep['macro_f1']:.4f} | macro-P {rep['macro_p']:.4f} | macro-R {rep['macro_r']:.4f} | "
          f"acc {rep['accuracy']:.4f} | n={rep['n']}")
    print(f"{'label':>24} {'P':>6} {'R':>6} {'F1':>6} {'gold':>6} {'pred':>6}")
    for l, v in rep["per_label"].items():
        print(f"{l:>24} {v['p']:6.3f} {v['r']:6.3f} {v['f1']:6.3f} {v['support']:6d} {v['predicted']:6d}")
    print("most frequent confusions (gold -> predicted):", ", ".join(f"{a}->{b} ({c})" for a, b, c in rep["top_confusions"][:6]))


# ====================================================================== bias tuning for macro-F1
def decode(lp, bias=None, crf=None):
    e = lp if bias is None else lp + bias
    if crf is None:
        return e.argmax(-1)
    return np.array(viterbi_np(e, *crf), dtype=np.int64)


def tune_bias(lps, ys, C, crf=None, grid=None, passes=3):
    """Coordinate ascent on per-class additive logit biases to maximise macro-F1 (lps/ys: lists per document)."""
    grid = np.round(np.arange(-3.0, 3.01, 0.25), 2) if grid is None else grid
    Y = np.concatenate(ys)
    if crf is None:
        LP = np.concatenate(lps)
        score = lambda b: macro_f1(Y, (LP + b).argmax(1), C)
    else:
        score = lambda b: macro_f1(Y, np.concatenate([decode(lp, b, crf) for lp in lps]), C)
    b = np.zeros(C)
    best = score(b)
    for _ in range(passes):
        improved = False
        for c in range(C):
            for g in grid:
                if g == b[c]:
                    continue
                b2 = b.copy(); b2[c] = g
                f = score(b2)
                if f > best + 1e-9:
                    best, b, improved = f, b2, True
        if not improved:
            break
    return b, best


def honest_bias_estimate(lps, ys, C, crf=None, seed=0):
    """2-fold over documents: tune biases on one half, measure on the other. Shows whether the bias gain is real."""
    idx = list(range(len(lps)))
    random.Random(seed).shuffle(idx)
    h = len(idx) // 2
    folds = [idx[:h], idx[h:]]
    before, after = [], []
    for a, b in ((0, 1), (1, 0)):
        if not folds[a] or not folds[b]:
            continue
        bias, _ = tune_bias([lps[i] for i in folds[a]], [ys[i] for i in folds[a]], C, crf)
        Yb = np.concatenate([ys[i] for i in folds[b]])
        before.append(macro_f1(Yb, np.concatenate([decode(lps[i], None, crf) for i in folds[b]]), C))
        after.append(macro_f1(Yb, np.concatenate([decode(lps[i], bias, crf) for i in folds[b]]), C))
    return (float(np.mean(before)), float(np.mean(after))) if before else (float("nan"), float("nan"))


# ====================================================================== training
def class_weights(docs, C, power):
    cnt = np.bincount(np.concatenate([d.y for d in docs]), minlength=C).astype(float)
    w = np.where(cnt > 0, (cnt.sum() / (C * np.maximum(cnt, 1))) ** power, 1.0)
    w = w / w[cnt > 0].mean()
    return np.clip(w, 0.2, 10.0), cnt


def steps_per_epoch(train_docs, cfg):
    n = len(epoch_windows(train_docs, cfg["max_sents"], cfg["chunk_tok"], random.Random(0)))
    return math.ceil(n / cfg["accum"])


@torch.no_grad()
def predict_doc(model, d, chunk_tok, amp):
    model.eval()
    if len(d.sents) == 0:
        return np.zeros((0, model.out[1].out_features), dtype=np.float32)
    with amp():
        em = model(Window(d, 0, len(d.sents), chunk_tok))
    return em.float().log_softmax(-1).cpu().numpy()


def predict_docs(model, docs, chunk_tok, amp):
    return {d.doc_id: predict_doc(model, d, chunk_tok, amp) for d in docs}


def evaluate(model, docs, labels, chunk_tok, amp, bias=None):
    C = len(labels)
    crf = model.crf.params_np() if model.crf is not None else None
    lps = predict_docs(model, docs, chunk_tok, amp)
    y = np.concatenate([d.y for d in docs])
    p = np.concatenate([decode(lps[d.doc_id], bias, crf) for d in docs if len(d.sents)])
    return report(y, p, labels), lps


def train_rr(model, train_docs, dev_docs, labels, cfg, device, amp, amp_dtype, teacher=None, ada=None, log=print):
    """cfg keys: max_sents chunk_tok epochs patience lr head_lr wd warmup accum grad_clip label_smooth class_w
       kd_alpha kd_T seed log_every. teacher: {doc_id: (n, C) teacher log-probs} or None."""
    C = len(labels)
    rng = random.Random(cfg["seed"])
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    bbp = [p for n, p in named if n.startswith("bb.")]
    hp = [p for n, p in named if not n.startswith("bb.")]
    groups = []
    if bbp:
        groups.append(dict(params=bbp, lr=cfg["lr"], base_lr=cfg["lr"], weight_decay=cfg["wd"]))
    groups.append(dict(params=hp, lr=cfg["head_lr"], base_lr=cfg["head_lr"], weight_decay=cfg["wd"]))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    scaler = make_scaler(device, amp_dtype)
    spe = steps_per_epoch(train_docs, cfg)
    total = spe * cfg["epochs"]
    warm = max(1, int(cfg["warmup"] * total))
    cw = torch.tensor(cfg["class_w"], dtype=torch.float32, device=device)
    keep_keys = {n for n, _ in named} | {n for n, _ in model.named_buffers() if "rank_mask" in n}
    tot, tr = count_params(model)
    log(f"{len(train_docs)} train docs | ~{spe} optimiser steps/epoch | {total} steps max | "
        f"params total {tot / 1e6:.2f}M, trainable {tr / 1e6:.3f}M ({100 * tr / max(1, tot):.2f}%)")
    history, best_f1, best_state, bad, step, t0 = [], -1.0, None, 0, 0, time.time()
    for ep in range(cfg["epochs"]):
        model.train()
        wins = epoch_windows(train_docs, cfg["max_sents"], cfg["chunk_tok"], rng)
        run, nrun = 0.0, 0
        for wi, win in enumerate(wins):
            y = torch.as_tensor(win.doc.y[win.i0:win.i1], device=device)
            with amp():
                em = model(win)
            em = em.float()
            if model.crf is not None:
                loss = model.crf.nll(em, y) / len(y)
            else:
                loss = F.cross_entropy(em, y, weight=cw, label_smoothing=cfg["label_smooth"])
            if teacher is not None:
                T = cfg["kd_T"]
                t = torch.as_tensor(teacher[win.doc.doc_id][win.i0:win.i1], device=device).float()
                kl = F.kl_div((em / T).log_softmax(-1), (t / T).log_softmax(-1), log_target=True,
                              reduction="batchmean") * T * T
                loss = cfg["kd_alpha"] * loss + (1 - cfg["kd_alpha"]) * kl
            if ada is not None:
                loss = loss + ada.reg_loss()
            scaler.scale(loss / cfg["accum"]).backward()
            run += loss.item(); nrun += 1
            if (wi + 1) % cfg["accum"] == 0 or wi == len(wins) - 1:
                mult = (step + 1) / warm if step < warm else max(0.0, 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm)))))
                for g in opt.param_groups:
                    g["lr"] = g["base_lr"] * mult
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for _, p in named], cfg["grad_clip"])
                if ada is not None:
                    ada.update(step)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                step += 1
                if step % cfg["log_every"] == 0:
                    log(f"  ep {ep + 1} step {step}/{total} loss {run / max(1, nrun):.4f} ({(time.time() - t0) / 60:.1f} min)")
        rec = dict(epoch=ep + 1, train_loss=run / max(1, nrun), minutes=(time.time() - t0) / 60)
        rep, _ = evaluate(model, dev_docs, labels, cfg["chunk_tok"], amp)
        rec.update(dev_macro_f1=rep["macro_f1"], dev_acc=rep["accuracy"])
        if rep["macro_f1"] > best_f1:
            best_f1, bad = rep["macro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep_keys}
            rec["best"] = True
        else:
            bad += 1
        history.append(rec)
        log(f"== epoch {ep + 1}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
        if bad >= cfg["patience"]:
            log(f"early stopping (no dev improvement for {bad} epochs)")
            break
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        log(f"restored best epoch (dev macro-F1 {best_f1:.4f})")
    return history, best_f1


# ====================================================================== task-adaptive pre-training (optional)
def tapt(bb, docs, epochs, lr, micro_bs, device, amp, amp_dtype, log=print):
    """Continued LM pre-training of the Phase-1 model on the UNLABELLED text of the RR train/dev documents
    (same objective as Phase 1: next-token for decoders, prefix-LM for the enc-dec). Uses only official data."""
    ids = []
    for d in docs:
        for t in d.tok:
            ids.extend(t)
        ids.append(EOS_ID)
    BL = bb.block_len
    n = len(ids) // BL
    if n < 2:
        log("TAPT skipped: not enough text"); return
    data = torch.tensor(ids[:n * BL], dtype=torch.long).view(n, BL)
    for p in bb.parameters():
        p.requires_grad = True
    bb.to(device).train()
    opt = torch.optim.AdamW(bb.parameters(), lr=lr, weight_decay=0.01, betas=(0.9, 0.95))
    scaler = make_scaler(device, amp_dtype)
    total = epochs * math.ceil(n / micro_bs)
    step, t0 = 0, time.time()
    log(f"TAPT: {n} blocks of {BL} tokens ({n * BL / 1e6:.2f}M tokens) x {epochs} epochs = {total} steps")
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, micro_bs):
            for g in opt.param_groups:
                g["lr"] = lr * min(1.0, (step + 1) / 20) * 0.5 * (1 + math.cos(math.pi * step / max(1, total)))
            batch = data[perm[i:i + micro_bs]].to(device)
            with amp():
                loss = bb.training_loss(batch)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(bb.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1
            if step % 25 == 0:
                log(f"  TAPT step {step}/{total} loss {loss.item():.4f} ({(time.time() - t0) / 60:.1f} min)")
    bb.eval()
    del opt
    if device == "cuda":
        torch.cuda.empty_cache()


# ====================================================================== outputs
def dump_probs(path, run_name, labels, docs, lps):
    out = {"run": run_name, "labels": labels, "docs": {}}
    for d in docs:
        out["docs"][d.doc_id] = {"domain": DOMAINS[d.domain], "n": len(d.sents),
                                 "logp": np.round(lps[d.doc_id], 4).tolist(),
                                 "gold": None if d.y is None else d.y.tolist()}
    json.dump(out, open(path, "w"))


def load_probs(path):
    j = json.load(open(path))
    return j["labels"], {k: np.asarray(v["logp"], dtype=np.float32) for k, v in j["docs"].items()}, j


def write_prediction(pred_ids, test_docs, labels, json_path, zip_path):
    out = {}
    for d in test_docs:
        p = [labels[int(i)] for i in pred_ids[d.doc_id]]
        assert len(p) == len(d.sents), f"{d.doc_id}: {len(p)} predictions for {len(d.sents)} sentences"
        out[d.doc_id] = p
    json.dump(out, open(json_path, "w"), indent=1)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, arcname="prediction.json")
    return out


@torch.no_grad()
def benchmark_sents_per_s(model, docs, chunk_tok, amp, n_docs=3):
    sel = sorted(docs, key=lambda d: -len(d.sents))[:n_docs]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t = time.time()
    n = 0
    for d in sel:
        predict_doc(model, d, chunk_tok, amp); n += len(d.sents)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return n / max(1e-6, time.time() - t)

In [ ]:
%%writefile rr_peft.py
"""Parameter-efficient adaptation methods for Phase 3, written from scratch (no `peft` library).
apply(method, bb, cfg) freezes nothing itself (the notebook freezes the backbone first) and returns an info dict.
  lora    : W0 x + (alpha/r) B A x                               (Hu et al., 2021)
  rslora  : same, scale alpha/sqrt(r)                            (Kalajdzievski, 2023)
  lorma   : (I + (alpha/r) B A) W0 x   multiplicative            (Bihany, Patel, Modi, 2025)   [compulsory]
  ia3     : learned rescaling of keys, values and FFN activations (Liu et al., 2022)
  adalora : SVD-form P diag(lambda) Q, importance-based rank pruning + orthogonality reg (Zhang et al., 2023)
  prompt  : n learned virtual tokens prepended to every input     (Lester et al., 2021)
  qlora   : frozen weights stored as 4-bit NormalFloat (blockwise absmax) + LoRA on top (Dettmers et al., 2023)
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rr_lib import replace_linears, replace_modules

DEFAULT_TARGETS = ("wq", "wk", "wv", "wo", "wg", "w_dkv", "w_ukv", "w_gate", "w_up", "w_down")


# ---------------------------------------------------------------- LoRA / rsLoRA
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0, rs=False):
        super().__init__()
        self.base, self.r = base, r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / math.sqrt(r) if rs else alpha / r
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


# ---------------------------------------------------------------- LoRMA (multiplicative)
class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        assert variant in ("plus", "pi")
        d = base.out_features
        self.base, self.r, self.variant = base, r, variant
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        if variant == "plus":
            self.scale = alpha / r
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))          # B = 0  ->  I + s BA = I at init
        else:
            self.scale = 1.0
            nn.init.normal_(self.B, std=0.02)
            nn.init.zeros_(self.A)
            with torch.no_grad():
                self.B[:, 0] = 1.0
                self.A[0, 0] = 1.0                                      # pi(BA) = I at init

    def pi(self):
        BA = self.B @ self.A
        d = BA.size(0)
        ar = torch.arange(d, device=BA.device)
        return BA.gather(1, (ar[None, :] - ar[:, None]) % d)

    def multiplier(self):
        if self.variant == "plus":
            return torch.eye(self.B.size(0), device=self.B.device, dtype=self.B.dtype) + self.scale * (self.B @ self.A)
        return self.pi()

    def forward(self, x):
        h = self.base(x)
        if self.variant == "plus":
            return h + F.linear(F.linear(h, self.A), self.B) * self.scale
        return F.linear(h, self.pi())


# ---------------------------------------------------------------- IA3
class IA3Out(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.in_features, self.out_features = base.in_features, base.out_features
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.in_features, self.out_features = base.in_features, base.out_features
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


# ---------------------------------------------------------------- AdaLoRA
class AdaLoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r = base, r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))                         # delta W = 0 at init
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.register_buffer("rank_mask", torch.ones(r))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.Q) * (self.lam * self.rank_mask), self.P) * self.scale

    def orth_loss(self):
        I = torch.eye(self.r, device=self.P.device)
        return ((self.P.t() @ self.P - I) ** 2).sum() + ((self.Q @ self.Q.t() - I) ** 2).sum()


class AdaLoRAController:
    """Budget schedule + sensitivity-based importance (EMA of |w * grad| times its uncertainty), as in AdaLoRA.
    Call reg_loss() inside the loss and update(step) after scaler.unscale_() and before optimizer.step()."""

    def __init__(self, model, init_r, target_r, total_steps, t_init_frac=0.1, t_final_frac=0.2, delta_t=5,
                 beta1=0.85, beta2=0.85, orth_reg=0.1):
        self.layers = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        assert self.layers, "no AdaLoRA layers found"
        self.total, self.dt = total_steps, delta_t
        self.ti, self.tf = int(t_init_frac * total_steps), int(t_final_frac * total_steps)
        self.b0, self.bT = init_r * len(self.layers), target_r * len(self.layers)
        self.b1, self.b2, self.orth = beta1, beta2, orth_reg
        self.ipt = [None] * len(self.layers)
        self.unc = [None] * len(self.layers)
        self.log = []

    def reg_loss(self):
        return self.orth * sum(m.orth_loss() for m in self.layers) / len(self.layers)

    def budget(self, step):
        end = self.total - self.tf
        if step < self.ti:
            return self.b0
        if step >= end:
            return self.bT
        prog = (step - self.ti) / max(1, end - self.ti)
        return int(self.bT + (self.b0 - self.bT) * (1 - prog) ** 3)

    @torch.no_grad()
    def update(self, step):
        scores = []
        for i, m in enumerate(self.layers):
            g = lambda p: p.grad if p.grad is not None else torch.zeros_like(p)
            s = torch.stack([(m.lam * g(m.lam)).abs(), (m.P * g(m.P)).abs().mean(0), (m.Q * g(m.Q)).abs().mean(1)]).float()
            s = torch.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)
            if self.ipt[i] is None:
                self.ipt[i], self.unc[i] = torch.zeros_like(s), torch.zeros_like(s)
            self.ipt[i] = self.b1 * self.ipt[i] + (1 - self.b1) * s
            self.unc[i] = self.b2 * self.unc[i] + (1 - self.b2) * (s - self.ipt[i]).abs()
            scores.append((self.ipt[i] * self.unc[i]).sum(0))
        end = self.total - self.tf
        if step < self.ti or step > end or ((step - self.ti) % self.dt != 0 and step != end):
            return
        b = self.budget(step)
        allsc = torch.cat(scores)
        if b >= allsc.numel():
            return
        thr = torch.topk(allsc, b).values[-1]
        for m, sc in zip(self.layers, scores):
            m.rank_mask.copy_((sc >= thr).float())
        self.log.append((step, b))

    def ranks(self):
        return [int(m.rank_mask.sum().item()) for m in self.layers]


# ---------------------------------------------------------------- Prompt tuning
class PromptEmbedding(nn.Module):
    """Token ids >= V are virtual prompt tokens with their own learned vectors (initialised from real words)."""

    def __init__(self, base_emb, n_prompt, seed=0):
        super().__init__()
        self.base, self.V, self.n = base_emb, base_emb.num_embeddings, n_prompt
        g = torch.Generator().manual_seed(seed)
        init_ids = torch.randint(100, min(5000, self.V), (n_prompt,), generator=g)
        self.prompt = nn.Parameter(base_emb.weight[init_ids].detach().clone())
        self.num_embeddings = self.V + n_prompt

    def forward(self, ids):
        is_p = ids >= self.V
        e = self.base(ids.clamp(max=self.V - 1))
        if bool(is_p.any()):
            pe = self.prompt[(ids - self.V).clamp(min=0, max=self.n - 1)]
            e = torch.where(is_p[..., None], pe.to(e.dtype), e)
        return e


# ---------------------------------------------------------------- 4-bit NormalFloat storage (QLoRA)
NF4 = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
                    -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
                    0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
                    0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen linear layer whose weight is stored as packed 4-bit NF4 codes (2 per byte) + fp16 absmax per block.
    The weight is de-quantised on the fly in every forward pass."""

    def __init__(self, lin, block=64):
        super().__init__()
        W = lin.weight.data.float()
        self.shape, self.numel, self.block = tuple(W.shape), W.numel(), block
        self.in_features, self.out_features = lin.in_features, lin.out_features
        flat = W.reshape(-1)
        pad = (-flat.numel()) % block
        if pad:
            flat = torch.cat([flat, flat.new_zeros(pad)])
        blocks = flat.view(-1, block)
        absmax = blocks.abs().max(1).values.clamp(min=1e-8)
        mids = (NF4[1:] + NF4[:-1]) / 2
        codes = torch.bucketize((blocks / absmax[:, None]).contiguous(), mids.to(blocks.device)).to(torch.uint8)
        self.register_buffer("packed", codes[:, 0::2] | (codes[:, 1::2] << 4))
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("nf4", NF4.clone())
        self.bias = None if lin.bias is None else nn.Parameter(lin.bias.data.clone(), requires_grad=False)

    def dequant(self):
        codes = torch.stack([self.packed & 15, self.packed >> 4], -1).view(self.packed.size(0), -1)
        W = self.nf4[codes.long()] * self.absmax.float()[:, None]
        return W.view(-1)[:self.numel].view(self.shape)

    def forward(self, x):
        return F.linear(x, self.dequant().to(x.dtype) if x.dtype != torch.float32 else self.dequant(), self.bias)


# ---------------------------------------------------------------- entry point
def apply(method, bb, cfg):
    targets = cfg.get("targets", DEFAULT_TARGETS)
    info = {"method": method}
    if method in ("lora", "rslora"):
        n = replace_linears(bb, targets, lambda l: LoRALinear(l, cfg["r"], cfg["alpha"], cfg.get("dropout", 0.0),
                                                              rs=(method == "rslora")))
    elif method == "lorma":
        n = replace_linears(bb, targets, lambda l: LoRMALinear(l, cfg["r"], cfg["alpha"], cfg.get("variant", "plus")))
    elif method == "ia3":
        n = replace_linears(bb, cfg.get("out_targets", ("wk", "wv", "w_ukv")), IA3Out)
        n += replace_linears(bb, cfg.get("in_targets", ("w_down",)), IA3In)
    elif method == "adalora":
        n = replace_linears(bb, targets, lambda l: AdaLoRALinear(l, cfg["init_r"], cfg["alpha"]))
    elif method == "prompt":
        bb.tok_emb = PromptEmbedding(bb.tok_emb, cfg["n_prompt"])
        bb._n_prompt, bb._prompt_base = cfg["n_prompt"], bb.tok_emb.V
        n = cfg["n_prompt"]
    elif method == "qlora":
        n = replace_linears(bb, targets, lambda l: LoRALinear(NF4Linear(l, cfg.get("block", 64)), cfg["r"],
                                                              cfg["alpha"], cfg.get("dropout", 0.0)))
        info["nf4_other"] = replace_modules(bb, lambda nm, m: isinstance(m, nn.Linear), lambda l: NF4Linear(l, cfg.get("block", 64)))
    else:
        raise ValueError(f"unknown method {method}")
    assert n > 0, f"{method}: no target modules matched - check the layer names of this backbone"
    info["n_modules"] = n
    print(f"[{method}] applied to {n} modules", {k: v for k, v in cfg.items() if k != "targets"})
    return info

## 5. Data: download, parse, tokenise, label statistics

In [ ]:
import rr_lib, rr_peft
importlib.reload(rr_lib); importlib.reload(rr_peft)
from huggingface_hub import hf_hub_download, HfApi
from tokenizers import Tokenizer
amp, AMP_DTYPE = rr_lib.get_amp(device)

files = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
p3 = [f for f in files if "Phase_3" in f]
print("Phase-3 files:", p3)
def fetch(name):
    m = [f for f in p3 if f.endswith("/" + name) or f.endswith(name)]
    assert m, f"{name} not found; Phase-3 files: {p3}"
    return hf_hub_download(HF_DATASET_REPO, m[0], repo_type="dataset", token=HF_TOKEN)

LABELS = rr_lib.load_label_vocab(json.load(open(fetch("label_vocab.json"), encoding="utf-8")))
assert len(LABELS) == 13, f"expected 13 labels, got {LABELS}"
L2I = {l: i for i, l in enumerate(LABELS)}
C = len(LABELS)
print("labels:", LABELS)

split = {}
for dom_i, dom in enumerate(rr_lib.DOMAINS):
    for part in ("train", "dev", "test"):
        docs, raw = rr_lib.parse_rr_file(fetch(f"{dom}_{part}.json"), dom_i, LABELS)
        split[(dom, part)] = docs
        print(f"{dom}_{part}: {len(docs)} docs, {sum(len(d.sents) for d in docs)} sentences | raw type {type(raw).__name__} | "
              f"example: id={docs[0].doc_id!r}, first sentence={docs[0].sents[0][:80]!r}, label={docs[0].labels[0] if docs[0].labels else None}")
train_docs = split[("CL", "train")] + split[("IT", "train")]
dev_docs   = split[("CL", "dev")]   + split[("IT", "dev")]
test_docs  = split[("CL", "test")]  + split[("IT", "test")]
assert len({d.doc_id for d in test_docs}) == len(test_docs), "duplicate test document IDs across CL/IT"
for d in train_docs + dev_docs:
    assert d.labels is not None, f"{d.doc_id}: no labels in a train/dev file"
    bad = set(d.labels) - set(LABELS)
    assert not bad, f"{d.doc_id}: unknown labels {bad}"

In [ ]:
# ---- Phase-1 export (backbone weights + tokenizer). For KD the same export is the source of the student.
export_dir = Path(P1_EXPORT_DIR) if P1_EXPORT_DIR else rr_lib.find_p1_export(BACKBONE_ARCH)
assert export_dir is not None, f"export_{BACKBONE_ARCH}/ not found - attach your Phase-1 output as a Kaggle input or set P1_EXPORT_DIR"
tokenizer = Tokenizer.from_file(str(export_dir / "tokenizer.json"))
TOK_HASH = rr_lib.file_hash(export_dir / "tokenizer.json")
for docs in (train_docs, dev_docs, test_docs):
    ntr = rr_lib.tokenize_docs(docs, tokenizer, MAX_SENT_TOK, L2I, NORMALIZE_WHITESPACE)
    print(f"tokenised {len(docs)} docs | sentences truncated to {MAX_SENT_TOK} tokens: {ntr}")
sl = np.array([len(t) for d in train_docs for t in d.tok])
print(f"tokens/sentence: mean {sl.mean():.1f} | p95 {np.percentile(sl, 95):.0f} | max {sl.max()} | "
      f"sentences/doc: mean {np.mean([len(d.sents) for d in train_docs]):.0f}, max {max(len(d.sents) for d in train_docs)}")

CLASS_W, CNT = rr_lib.class_weights(train_docs, C, CLASS_WEIGHT_POW)
dev_cnt = np.bincount(np.concatenate([d.y for d in dev_docs]), minlength=C)
print(f"\n{'label':>24} {'train':>7} {'dev':>6} {'weight':>7}")
for i, l in enumerate(LABELS):
    print(f"{l:>24} {int(CNT[i]):7d} {int(dev_cnt[i]):6d} {CLASS_W[i]:7.2f}")
for dom in rr_lib.DOMAINS:
    y = np.concatenate([d.y for d in split[(dom, "train")]])
    print(dom, "train label distribution:", {LABELS[k]: int(v) for k, v in enumerate(np.bincount(y, minlength=C)) if v})

## 6. Build the model (Phase-1 backbone -> [TAPT] -> PEFT / KD student -> hierarchical head)

In [ ]:
arch_mod, bcfg, bb = rr_lib.load_p1(export_dir)
P1_PARAMS = sum(p.numel() for p in bb.parameters())
print(f"Phase-1 '{BACKBONE_ARCH}' loaded from {export_dir}: {P1_PARAMS / 1e6:.1f}M params, kind={bb.kind}")

TEACHER = None
if METHOD == "kd":
    # ---- teacher = average of the train-set probabilities of earlier runs (attach their output folders as inputs)
    tdirs = MCFG["teacher_dirs"] or sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/probs_train.json", recursive=True)})
    tdirs = [t for t in tdirs if Path(t).resolve() != WORK.resolve()]
    assert tdirs, "KD needs at least one earlier run folder with probs_train.json (attach it as a Kaggle input)"
    acc, TEACHER_INFO = None, []
    for t in tdirs:
        labs, lp, j = rr_lib.load_probs(Path(t) / "probs_train.json")
        assert labs == LABELS, f"label order differs in {t}"
        probs = {k: np.exp(v) for k, v in lp.items()}
        acc = probs if acc is None else {k: acc[k] + probs[k] for k in acc}
        r = Path(t) / "results.json"
        TEACHER_INFO.append(dict(dir=t, run=j["run"], **({k: json.load(open(r)).get(k) for k in ("dev_macro_f1", "params_total")} if r.exists() else {})))
    TEACHER = {k: np.log(v / len(tdirs) + 1e-8).astype(np.float32) for k, v in acc.items()}
    missing = [d.doc_id for d in train_docs if d.doc_id not in TEACHER]
    assert not missing, f"teacher has no probabilities for train docs {missing[:5]}"
    print("teacher = average of", len(tdirs), "runs:", [t["run"] for t in TEACHER_INFO])
    # ---- student: every other layer of the Phase-1 model (initialised from its weights)
    nL = len(rr_lib.blocks_of(bb))
    keep = MCFG["keep"] or list(range(0, nL, 2))
    bb, bcfg = rr_lib.make_student_bb(arch_mod, bcfg, bb, keep)
    print(f"student keeps layers {keep} of {nL}: {sum(p.numel() for p in bb.parameters()) / 1e6:.1f}M params")

if TAPT:
    rr_lib.tapt(bb, train_docs + dev_docs, TAPT_EPOCHS, TAPT_LR, TAPT_MICRO_BS, device, amp, AMP_DTYPE)

rr_lib.strip_backbone(bb)
print("encoder padding-mask patch applied to", rr_lib.patch_encoder_padding_mask(bb), "layers")
if METHOD == "kd":
    for p in bb.parameters():
        p.requires_grad = True
    PEFT_INFO = {"method": "kd", "student_layers": keep}
else:
    for p in bb.parameters():
        p.requires_grad = False
    PEFT_INFO = rr_peft.apply(METHOD, bb, MCFG)

# ---- context length check (decoders read [prompt, chunk, chunk])
lim = rr_lib.max_positions(bb)
need = lambda ct: getattr(bb, "_n_prompt", 0) + (2 * ct if bb.kind == "decoder" else ct)
while need(CHUNK_TOK) > lim:
    CHUNK_TOK -= 32
assert MAX_SENT_TOK <= CHUNK_TOK
print(f"position limit {lim} | CHUNK_TOK {CHUNK_TOK} -> backbone sequence length {need(CHUNK_TOK)}")

model = rr_lib.RRModel(bb, C, HCFG).to(device)
tot, tr = rr_lib.count_params(model)
print(f"RR model: {tot / 1e6:.2f}M params | trainable {tr / 1e6:.3f}M ({100 * tr / tot:.2f}%)")

## 7. Train (early stopping on dev macro-F1)

In [ ]:
TCFG = dict(max_sents=MAX_SENTS, chunk_tok=CHUNK_TOK, epochs=EPOCHS, patience=PATIENCE, lr=MCFG["lr"], head_lr=HEAD_LR,
            wd=WD, warmup=WARMUP, accum=ACCUM, grad_clip=GRAD_CLIP, label_smooth=LABEL_SMOOTH, class_w=CLASS_W,
            kd_alpha=MCFG.get("alpha", 1.0), kd_T=MCFG.get("T", 1.0), seed=SEED, log_every=10)
ADA = None
if METHOD == "adalora":
    total_steps = rr_lib.steps_per_epoch(train_docs, TCFG) * EPOCHS
    ADA = rr_peft.AdaLoRAController(model, MCFG["init_r"], MCFG["target_r"], total_steps,
                                    delta_t=MCFG["delta_t"], orth_reg=MCFG["orth_reg"])
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
t_train = time.time()
history, best_f1 = rr_lib.train_rr(model, train_docs, dev_docs, LABELS, TCFG, device, amp, AMP_DTYPE,
                                   teacher=TEACHER, ada=ADA)
TRAIN_MIN = (time.time() - t_train) / 60
PEAK_GB = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
print(f"training {TRAIN_MIN:.1f} min | peak GPU memory {PEAK_GB:.1f} GB | best dev macro-F1 {best_f1:.4f}")
if ADA is not None:
    print("AdaLoRA final ranks per adapted matrix:", ADA.ranks())

## 8. Dev evaluation: per-label, per-domain, bias tuning (with an honest 2-fold estimate)

In [ ]:
CRF_NP = model.crf.params_np() if model.crf is not None else None
dev_rep, dev_lp = rr_lib.evaluate(model, dev_docs, LABELS, CHUNK_TOK, amp)
rr_lib.print_report(dev_rep, "DEV (CL+IT, scored together like Codabench)")
dom_rep = {}
for dom in rr_lib.DOMAINS:
    ds = split[(dom, "dev")]
    y = np.concatenate([d.y for d in ds]); p = np.concatenate([rr_lib.decode(dev_lp[d.doc_id], None, CRF_NP) for d in ds])
    dom_rep[dom] = rr_lib.report(y, p, LABELS)
    print(f"  {dom} dev macro-F1 {dom_rep[dom]['macro_f1']:.4f} (labels present: {dom_rep[dom]['macro_f1_present_labels']:.4f}) | acc {dom_rep[dom]['accuracy']:.4f}")

lps = [dev_lp[d.doc_id] for d in dev_docs]; ys = [d.y for d in dev_docs]
BIAS = np.zeros(C)
if TUNE_BIAS:
    BIAS, f_tuned = rr_lib.tune_bias(lps, ys, C, CRF_NP)
    h0, h1 = rr_lib.honest_bias_estimate(lps, ys, C, CRF_NP, seed=SEED)
    print(f"\nbias tuning: dev macro-F1 {dev_rep['macro_f1']:.4f} -> {f_tuned:.4f} (tuned on dev itself)")
    print(f"honest 2-fold estimate (tune on half the docs, score the other half): {h0:.4f} -> {h1:.4f}")
    print("biases:", {LABELS[i]: float(b) for i, b in enumerate(BIAS) if b})
    USE_BIAS = h1 > h0          # keep the biased version as the main submission only if it also helps on held-out docs
else:
    f_tuned, h0, h1, USE_BIAS = dev_rep["macro_f1"], float("nan"), float("nan"), False
print("main submission uses bias:", USE_BIAS)

# files for the official local evaluator:  python evaluate_phase3.py --truth-cl dev_truth_CL.json --truth-it dev_truth_IT.json --prediction dev_prediction.json --labels labels.json
for dom in rr_lib.DOMAINS:
    json.dump({d.doc_id: d.labels for d in split[(dom, "dev")]}, open(WORK / f"dev_truth_{dom}.json", "w"))
json.dump({d.doc_id: [LABELS[i] for i in rr_lib.decode(dev_lp[d.doc_id], BIAS if USE_BIAS else None, CRF_NP)] for d in dev_docs},
          open(WORK / "dev_prediction.json", "w"))
json.dump(LABELS, open(WORK / "labels.json", "w"))

## 9. Test predictions -> `submission.zip`, probabilities for the ensemble / KD, results

In [ ]:
RUN_NAME = WORK.name
test_lp = rr_lib.predict_docs(model, test_docs, CHUNK_TOK, amp)
pred_main = {d.doc_id: rr_lib.decode(test_lp[d.doc_id], BIAS if USE_BIAS else None, CRF_NP) for d in test_docs}
pred_alt  = {d.doc_id: rr_lib.decode(test_lp[d.doc_id], None if USE_BIAS else BIAS, CRF_NP) for d in test_docs}
out = rr_lib.write_prediction(pred_main, test_docs, LABELS, WORK / "prediction.json", WORK / "submission.zip")
rr_lib.write_prediction(pred_alt, test_docs, LABELS, WORK / "prediction_alt.json", WORK / "submission_alt.zip")
from collections import Counter
print(f"{len(out)} test docs | {sum(map(len, out.values()))} sentences | predicted label counts:",
      dict(Counter(l for v in out.values() for l in v).most_common()))
print("main:", WORK / "submission.zip", "| alternative (", "no bias" if USE_BIAS else "with bias", "):", WORK / "submission_alt.zip")

# ---- probabilities (ensemble notebook + teacher for the KD run). Train probs only for non-KD runs.
rr_lib.dump_probs(WORK / "probs_dev.json", RUN_NAME, LABELS, dev_docs, dev_lp)
rr_lib.dump_probs(WORK / "probs_test.json", RUN_NAME, LABELS, test_docs, test_lp)
if METHOD != "kd":
    rr_lib.dump_probs(WORK / "probs_train.json", RUN_NAME, LABELS, train_docs, rr_lib.predict_docs(model, train_docs, CHUNK_TOK, amp))

sps = rr_lib.benchmark_sents_per_s(model, dev_docs, CHUNK_TOK, amp)
tot, tr = rr_lib.count_params(model)
res = dict(run=RUN_NAME, method=METHOD, backbone=BACKBONE_ARCH, tapt=TAPT, peft=PEFT_INFO, mcfg=MCFG, hcfg=HCFG,
           chunk_tok=CHUNK_TOK, max_sents=MAX_SENTS, params_total=tot, params_trainable=tr, phase1_params=P1_PARAMS,
           train_min=TRAIN_MIN, peak_gpu_gb=PEAK_GB, infer_sents_per_s=sps,
           dev_macro_f1=dev_rep["macro_f1"], dev_macro_f1_bias_tuned=f_tuned, honest_bias=[h0, h1], used_bias=bool(USE_BIAS),
           bias=BIAS.tolist(), dev=dev_rep, dev_by_domain=dom_rep, history=history, tokenizer_hash=TOK_HASH,
           adalora_ranks=ADA.ranks() if ADA is not None else None,
           teacher=TEACHER_INFO if METHOD == "kd" else None)
json.dump(res, open(WORK / "results.json", "w"), indent=1, default=float)
torch.save({k: v.cpu() for k, v in model.state_dict().items()
            if k in {n for n, p in model.named_parameters() if p.requires_grad} or "rank_mask" in k}, WORK / "trainable_state.pt")
print(f"\nSUMMARY {RUN_NAME}: dev macro-F1 {dev_rep['macro_f1']:.4f} (bias-tuned {f_tuned:.4f}; honest {h0:.4f}->{h1:.4f}) | "
      f"CL {dom_rep['CL']['macro_f1']:.4f} IT {dom_rep['IT']['macro_f1']:.4f} | params {tot / 1e6:.1f}M, trainable {tr / 1e6:.3f}M | "
      f"train {TRAIN_MIN:.1f} min | {sps:.0f} sentences/s | peak {PEAK_GB:.1f} GB")
if METHOD == "kd":
    tf = [t.get("dev_macro_f1") for t in TEACHER_INFO if t.get("dev_macro_f1") is not None]
    tp = [t.get("params_total") for t in TEACHER_INFO if t.get("params_total")]
    if tf:
        print(f"KD: student keeps {100 * dev_rep['macro_f1'] / max(tf):.1f}% of the best teacher's dev macro-F1 ({max(tf):.4f})"
              + (f" with {100 * tot / max(tp):.0f}% of its parameters" if tp else ""))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot([h["epoch"] for h in history], [h["train_loss"] for h in history], "o-"); ax[0].set_title("train loss"); ax[0].grid(True)
ax[1].plot([h["epoch"] for h in history], [h["dev_macro_f1"] for h in history], "o-"); ax[1].set_title("dev macro-F1"); ax[1].grid(True)
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(WORK / "curves.png", dpi=150); plt.show()